# 08 — RANSAC: robust fitting, iteration number, several models

**Needed for the assignment:** $\bigstar$ YES. **Study:** 2 h, **Practice:** 1.5 h.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Idea

Least squares is ruined by a single outlier. **RANSAC** (Fischler & Bolles, 1981) instead **guesses** a model from a minimal random sample, **counts** how many points agree with it (distance $<$ threshold), keeps the best guess, and finally **refines** it with least squares on its inliers. The cost it minimises is a truncated least squares: inliers cost their squared error and outliers a constant.

## Key formulas

**Loop:**  sample $m$ points $\rightarrow$ fit $\rightarrow$ residuals of all points $\rightarrow$ inliers ($<\sigma$) $\rightarrow$ keep the best $\rightarrow$ repeat.\
**Minimal sample size:** line $m=2$, plane $3$, circle $3$, sphere $4$, homography $4$, fundamental matrix $7$ or $8$.\
**Iteration number** (inlier ratio $w=Q/N$, confidence $p$, e.g. $0.99$): $$P(\text{one sample all inliers})=w^m,\qquad P(k\text{ bad samples})=(1-w^m)^k\le1-p$$ $$\Rightarrow\ \boxed{k=\left\lceil\frac{\log(1-p)}{\log(1-w^m)}\right\rceil}$$ **Adaptive:** start with a large $k$. Each time a better model is found, recompute $k$ with the new $w$ and keep the smaller value.\
**Several models** (planes in a room):

-   *sequential*: find the best model, remove its inliers, run again on the rest;

-   *joint*: Lab 5 samples $2L$ points at once for $L$ lines.

# Worked example 1 (on paper): how many iterations?

|                          |                                                                   |                                    |
|:-------------------------|:------------------------------------------------------------------|:-----------------------------------|
| line, $m=2$, $w=0.5$:    | $\dfrac{\log0.01}{\log(1-0.25)}=\dfrac{-4.605}{-0.2877}=16.01$    | $\Rightarrow k=17$                 |
| plane, $m=3$, $w=0.25$:  | $\dfrac{-4.605}{\log(1-0.015625)}=\dfrac{-4.605}{-0.01575}=292.4$ | $\Rightarrow k=293$                |
| sphere, $m=4$, $w=0.05$: | $w^4=6.25\cdot10^{-6}$, $\dfrac{-4.605}{-6.25\cdot10^{-6}}$       | $\Rightarrow k\approx737\,000$ (!) |

The last line is the reason why, in the assignment, the sphere is *not* sampled from 4 random points of the whole cloud.

# Worked example 2 (on paper): two hypotheses by hand

Points: $(0,0),(1,1),(2,2),(3,3),(0,3),(3,0)$, threshold $\sigma=0.5$.

**Hypothesis 1** from $(0,0),(1,1)$: $\Delta=(1,1)$, normal $\tfrac1{\sqrt2}(-1,1)$, $c=0$. Line: $\tfrac{-x+y}{\sqrt2}=0$.\
Distances: $0,0,0,0,\ \tfrac3{\sqrt2}=2.12,\ 2.12$, so **4 inliers**.

**Hypothesis 2** from $(0,3),(3,0)$: $\Delta=(3,-3)$, normal $\tfrac1{\sqrt2}(1,1)$, $c=-\tfrac3{\sqrt2}$. Line: $\tfrac{x+y-3}{\sqrt2}=0$.\
Distances: $2.12,\ 0.71,\ 0.71,\ 2.12,\ 0,\ 0$, so **2 inliers**.

Hypothesis 1 wins. Then refine it with least squares (topic 06) on its 4 inliers. The 2 outliers had no influence at all.

# The same in Python / NumPy: sequential plane RANSAC (floor + wall + 30% outliers)

In [2]:
import math
import numpy as np

def iteration_number(confidence, sample_size, inlier_ratio):
    return math.ceil(math.log(1 - confidence) / math.log(1 - inlier_ratio**sample_size))

print("line   m=2, w=0.50:", iteration_number(0.99, 2, 0.50))
print("plane  m=3, w=0.25:", iteration_number(0.99, 3, 0.25))
print("sphere m=4, w=0.05:", iteration_number(0.99, 4, 0.05))

# ---------------- plane RANSAC (3D), sequential: find several planes ----------------
def fit_plane_three_points(p1, p2, p3):
    n = np.cross(p2 - p1, p3 - p1)
    norm = np.linalg.norm(n)
    if norm < 1e-9:
        return None
    n = n / norm
    return n, -n @ p1

def refine_plane(P):
    c = P.mean(axis=0)
    val, vec = np.linalg.eig((P - c).T @ (P - c))
    val, vec = val.real, vec.real
    n = vec[:, np.argmin(val)]
    return n, -n @ c

def ransac_plane(P, threshold, rng, confidence=0.99, max_iterations=1000):
    best_count, best_plane = 0, None
    it = 0
    while it < max_iterations:
        it += 1
        i = rng.choice(len(P), 3, replace=False)
        plane = fit_plane_three_points(*P[i])
        if plane is None:
            continue
        n, d = plane
        count = np.count_nonzero(np.abs(P @ n + d) < threshold)
        if count > best_count:
            best_count, best_plane = count, plane
            max_iterations = min(max_iterations,
                                 iteration_number(confidence, 3, best_count / len(P)))
    return best_plane, it

# synthetic scene: floor z=0, wall x=2, plus 30 % random outliers
rng = np.random.default_rng(0)
floor = np.column_stack([rng.uniform(0, 2, 400), rng.uniform(0, 3, 400), rng.normal(0, 0.01, 400)])
wall = np.column_stack([2 + rng.normal(0, 0.01, 300), rng.uniform(0, 3, 300), rng.uniform(0, 2, 300)])
outl = rng.uniform([0, 0, 0], [2, 3, 2], size=(300, 3))
P = np.vstack([floor, wall, outl])

remaining = P.copy()
for k in range(2):
    (n, d), iters = ransac_plane(remaining, 0.03, rng)
    inl = np.abs(remaining @ n + d) < 0.03
    n, d = refine_plane(remaining[inl])                  # least-squares refinement
    inl = np.abs(remaining @ n + d) < 0.03
    print(f"plane {k}: n={np.round(n, 3)}, d={d:.3f}, inliers={inl.sum()}, iterations={iters}")
    remaining = remaining[~inl]                           # remove and search again
print("left over:", len(remaining), "points")

line   m=2, w=0.50: 17
plane  m=3, w=0.25: 293
sphere m=4, w=0.05: 736825
plane 0: n=[-0.001 -0.     1.   ], d=0.001, inliers=405, iterations=69
plane 1: n=[-1.     0.001  0.001], d=1.999, inliers=300, iterations=35
left over: 295 points


## Practical notes (RANSAC slides, "Cons" and "Upgrades")

-   **Threshold $\sigma$** must be known. It depends on the sensor noise (the solution measures it from the data).

-   The method is slow when $w$ is low, because $k$ grows like $1/w^m$.

-   Not every all-inlier sample gives a good model (noise). That is why the best model is **refined** on all its inliers (locally optimised RANSAC).

-   Watch out for **degenerate samples**: collinear points for a plane, coplanar points for a sphere. Skip them.

## Where this is used in the assignment

This is the core of the assignment: `ransac_plane` and `ransac_sphere` use this loop with `iteration_number` from Lab 3. Planes are found *sequentially* (find, remove, repeat). For the sphere the solution changes only the *sampling*: one random seed point plus 3 points near it. That keeps $k$ small even though $w$ is tiny (example 1).

## Course / lab link

Lab 3 has single-line RANSAC with the adaptive iteration number. Lab 5 has multi-line RANSAC with a mask so that a point belongs to only one line. Note that Lab 5 rounds $k$ down (`int`) where Lab 3 rounds up (`ceil`); `ceil` is the correct one.

## Practice

Write your solution in the empty cell under each exercise. The **expected result** tells you what you should get, not how to get it. Grey *provided data* cells are already filled in; run them first.

### Exercise 1

Compute $k$ for $p=0.95$: line with $w=0.3$, plane with $w=0.6$, sphere with $w=0.5$.

> **Expected result:** $32,\ 13,\ 47$.

In [3]:
# Exercise 1: your solution


### Exercise 2

Run the 6-point toy example in NumPy: write `fit_line_two_points` and `distance`, and check the inlier counts of both hypotheses ($\sigma=0.5$).

> **Expected result:** Hypothesis 1 has 4 inliers, hypothesis 2 has 2 inliers.

In [4]:
# Exercise 2: your solution


### Exercise 3

**Warm-up for the assignment.** The provided cloud is a floor, a ball on it, and outliers. First find and remove the plane with plane RANSAC (threshold 0.02, refine). Then write `ransac_sphere` (4 random points, `fit_sphere` from topic 07, threshold 0.02) on the remaining points.

> **Expected result:** Plane: $\mathbf n\approx(0,0,1)$, $d\approx0$, about 1526 inliers; about 1104 points remain. Sphere: centre $\approx(0.50,-0.30,0.30)$, $r\approx0.300$, about 588 inliers (exact counts can differ a little with your random sampling).

In [5]:
# --- provided data (run me first) ---
# P : (2625, 3) points =
#     1500 floor points (z ~ 0, noise 0.005) in [-2,2] x [-2,2]
#     600 points on a ball: centre (0.5, -0.3, 0.3), radius 0.3, noise 0.005
#     525 random outliers in [-2,2] x [-2,2] x [0,1.5]
# Edge case: the ball touches the floor at one point (0.5, -0.3, 0).
rng = np.random.default_rng(3)
floor = np.column_stack([rng.uniform(-2, 2, 1500), rng.uniform(-2, 2, 1500), rng.normal(0, 0.005, 1500)])
d = rng.normal(size=(600, 3)); d /= np.linalg.norm(d, axis=1, keepdims=True)
ball = np.array([0.5, -0.3, 0.3]) + 0.3 * d + rng.normal(0, 0.005, (600, 3))
outl = rng.uniform([-2, -2, 0], [2, 2, 1.5], (525, 3))
P = np.vstack([floor, ball, outl])

In [6]:
# Exercise 3: your solution


### Exercise 4

In exercise 3, what goes wrong if the plane threshold is large (e.g. 0.1)? Count how many **ball** points are within 0.1 of the floor plane.

> **Expected result:** 103 ball points have $|z|<0.1$. The plane would steal the bottom cap of the ball, which is why the assignment solution finds the sphere first and checks normals.

In [7]:
# Exercise 4: your solution
